In [126]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nakulhemantkarpe/nashik-aqi/Nashik_AQIBulletins.csv


# Feature Extraction
now we have analyzed and understood the patterns and trends in the data, time to extract relavant features:

In [127]:
df=pd.read_csv("/kaggle/input/datasets/nakulhemantkarpe/nashik-aqi/Nashik_AQIBulletins.csv")

In [128]:
df.head()

,date,City,No. Stations,Air Quality,Index Value,Prominent Pollutant
0,2016-05-19,Nashik,1.0,Satisfactory,87,PM10
1,2016-05-23,Nashik,1.0,Satisfactory,62,PM10
2,2016-05-24,Nashik,1.0,Satisfactory,55,PM10
3,2016-05-25,Nashik,1.0,Good,44,PM10
4,2016-05-26,Nashik,1.0,Good,40,O3


In [129]:
df['date']=pd.to_datetime(df['date'])

In [130]:
df['year']=df['date'].dt.year

In [131]:
df['day']=df['date'].dt.day

In [132]:
df['month']=df['date'].dt.month

In [133]:
df['weekdays']=df['date'].dt.dayofweek

In [134]:
df.head()

,date,City,No. Stations,Air Quality,Index Value,Prominent Pollutant,year,day,month,weekdays
0,2016-05-19,Nashik,1.0,Satisfactory,87,PM10,2016,19,5,3
1,2016-05-23,Nashik,1.0,Satisfactory,62,PM10,2016,23,5,0
2,2016-05-24,Nashik,1.0,Satisfactory,55,PM10,2016,24,5,1
3,2016-05-25,Nashik,1.0,Good,44,PM10,2016,25,5,2
4,2016-05-26,Nashik,1.0,Good,40,O3,2016,26,5,3


# **Cylic Features**

now cyclc features are a standard practice, because :
* computer thinks 1st and 2 nd month are close to each other, which is true
* but it also thinks 12th month and 1st month are far apart from each other, which is not true in real world(similarly how the clock works)
* to make computer understand this real world logic, we create cylic features using:

* sine and cosine, think of it as a clock
**full circle=360 degrees**
**total 12 months**
so each month gets **360=2*pi radians/12=30 degree for 1 st month, 60 for second and so on**
* then we compute sin and cosine for them
* so now sin(360 * 12/12) and cos(360 * 12/12)=(1,0) and sin(360 * 1/12) and cos(360 * 1/12)=(0.866,0.5)
* **now computer can see they(december and january) are close**


In [135]:
df['month_sin']=np.sin(2*np.pi*df['month']/12)

In [136]:
df['month_cos']=np.cos(2*np.pi*df['month']/12)

In [137]:
df.head(100)

,date,City,No. Stations,Air Quality,Index Value,Prominent Pollutant,year,day,month,weekdays,month_sin,month_cos
0,2016-05-19,Nashik,1.0,Satisfactory,87,PM10,2016,19,5,3,0.500000,-0.866025
1,2016-05-23,Nashik,1.0,Satisfactory,62,PM10,2016,23,5,0,0.500000,-0.866025
2,2016-05-24,Nashik,1.0,Satisfactory,55,PM10,2016,24,5,1,0.500000,-0.866025
3,2016-05-25,Nashik,1.0,Good,44,PM10,2016,25,5,2,0.500000,-0.866025
4,2016-05-26,Nashik,1.0,Good,40,O3,2016,26,5,3,0.500000,-0.866025
...,...,...,...,...,...,...,...,...,...,...,...,...
95,2016-10-02,Nashik,1.0,Good,36,CO,2016,2,10,6,-0.866025,0.500000
96,2016-10-03,Nashik,1.0,Good,34,CO,2016,3,10,0,-0.866025,0.500000
97,2016-10-04,Nashik,1.0,Good,38,O3,2016,4,10,1,-0.866025,0.500000
98,2016-10-05,Nashik,1.0,Good,34,PM2.5,2016,5,10,2,-0.866025,0.500000


# Lag Features

we will create lag 1day and 7days feature to train model based on todays AQI to 1 day before and a week before, it helps in better prediction making

In [138]:
df['AQI_lag_1']=df['Index Value'].shift(1)

In [139]:
df['AQI_lag_week_ago']=df['Index Value'].shift(7)

In [140]:
df[['AQI_lag_1','AQI_lag_week_ago']].isnull().sum()

AQI_lag_1           1
AQI_lag_week_ago    7
dtype: int64

In [141]:
df.dropna(inplace=True)

removed the first 7 null values, so our dataset starts from 7 days ahead but we have lag values from a week ago

In [142]:
df


,date,City,No. Stations,Air Quality,Index Value,Prominent Pollutant,year,day,month,weekdays,month_sin,month_cos,AQI_lag_1,AQI_lag_week_ago
7,2016-05-29,Nashik,1.0,Good,39,PM10,2016,29,5,6,5.000000e-01,-0.866025,45.0,87.0
8,2016-05-30,Nashik,1.0,Good,36,O3,2016,30,5,0,5.000000e-01,-0.866025,39.0,62.0
9,2016-06-01,Nashik,1.0,Good,47,PM10,2016,1,6,2,1.224647e-16,-1.000000,36.0,55.0
10,2016-06-02,Nashik,1.0,Satisfactory,52,PM10,2016,2,6,3,1.224647e-16,-1.000000,47.0,44.0
11,2016-06-03,Nashik,1.0,Satisfactory,57,PM10,2016,3,6,4,1.224647e-16,-1.000000,52.0,40.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2407,2023-12-27,Nashik,4.0,Moderate,197,PM2.5,2023,27,12,2,-2.449294e-16,1.000000,188.0,101.0
2408,2023-12-28,Nashik,3.0,Moderate,199,PM2.5,2023,28,12,3,-2.449294e-16,1.000000,197.0,101.0
2409,2023-12-29,Nashik,4.0,Moderate,154,PM2.5,2023,29,12,4,-2.449294e-16,1.000000,199.0,117.0
2410,2023-12-30,Nashik,4.0,Moderate,170,PM2.5,2023,30,12,5,-2.449294e-16,1.000000,154.0,198.0


now we dont need date column since we have already extracted its features

In [143]:
df.drop('date', axis = 1, inplace=True)

In [144]:
df

,City,No. Stations,Air Quality,Index Value,Prominent Pollutant,year,day,month,weekdays,month_sin,month_cos,AQI_lag_1,AQI_lag_week_ago
7,Nashik,1.0,Good,39,PM10,2016,29,5,6,5.000000e-01,-0.866025,45.0,87.0
8,Nashik,1.0,Good,36,O3,2016,30,5,0,5.000000e-01,-0.866025,39.0,62.0
9,Nashik,1.0,Good,47,PM10,2016,1,6,2,1.224647e-16,-1.000000,36.0,55.0
10,Nashik,1.0,Satisfactory,52,PM10,2016,2,6,3,1.224647e-16,-1.000000,47.0,44.0
11,Nashik,1.0,Satisfactory,57,PM10,2016,3,6,4,1.224647e-16,-1.000000,52.0,40.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2407,Nashik,4.0,Moderate,197,PM2.5,2023,27,12,2,-2.449294e-16,1.000000,188.0,101.0
2408,Nashik,3.0,Moderate,199,PM2.5,2023,28,12,3,-2.449294e-16,1.000000,197.0,101.0
2409,Nashik,4.0,Moderate,154,PM2.5,2023,29,12,4,-2.449294e-16,1.000000,199.0,117.0
2410,Nashik,4.0,Moderate,170,PM2.5,2023,30,12,5,-2.449294e-16,1.000000,154.0,198.0


Our data is ready to be trained,now we extract it


In [145]:
df.to_csv("/kaggle/working/processed_aqi_data.csv", index=False)

In [146]:
import os

os.listdir("/kaggle/working")

['processed_aqi_data.csv', '.virtual_documents']